<a href="https://colab.research.google.com/github/Zikirullo/clear-vision/blob/develop/notebooks/09_pseudo_labels.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 09 · Pseudo-labels for missing COCO objects

Each source labels only its own classes: the people in rf_cone images are not boxed. That
teaches the model to ignore them, and in notebook 08 correct person and car detections in
those images counted as false positives (person 0.39, car 0.21 mAP50).

This notebook runs **yolo11x** (the most accurate COCO model) over every image of a
non-COCO source and **adds** boxes for the COCO classes in our list:

```
person, bicycle, car, motorcycle, bus, truck, dog, bench, fire_hydrant, traffic_light
```

- **Which images:** every source except `coco_subset` and `openimages`, which already
  label these classes. The source is the file name prefix. All splits are labeled, so the
  evaluation gets fairer too.
- **Which boxes are added:** only a box with confidence ≥ `CONF` that doesn't overlap an
  existing box of the same class with IoU > `IOU`.
- **Original labels are never removed or changed.** New lines are appended.

```
reads   MyDrive/clear_vision/datasets/merged.zip + merged.done   (prepare_dataset, unpacked to /content)
writes  MyDrive/clear_vision/datasets/merged.zip          re-zipped, uploaded and verified like append mode
        MyDrive/clear_vision/datasets/merged.done         written only after the verified upload
        MyDrive/clear_vision/datasets/merge_report.json   + pseudo_labels_added per class per source
        MyDrive/clear_vision/datasets/labels_original.zip original content of every changed label file
        MyDrive/clear_vision/datasets/pseudo_labels.csv   one row per added box
```

- **Upload safety:** the same steps as append mode.
  - It checks free space (+10%).
  - The backup and CSV are uploaded, flushed and verified first, so merged.zip is never
    changed without a working backup.
  - It deletes the old `merged.done`, uploads the zip, flushes Drive, and verifies the size
    and CRCs. Only then does it write `merged.done`.
  - If an upload fails, `merged.done` stays missing and training refuses the zip. Re-run
    this notebook, or notebook 02 if the zip on Drive is damaged.
- **Re-running:** it skips if `merge_report.json` already has pseudo-labels. With
  `FORCE_PSEUDO = True`, it first restores the original labels from
  `labels_original.zip`, then labels again. After appending new sources in notebook 02,
  re-run with `FORCE_PSEUDO = True`.
- **Runtime:** T4 GPU (Runtime → Change runtime type → T4 GPU). Run all cells.

In [ ]:
# Settings
REPO_URL = "https://github.com/Zikirullo/clear-vision.git"
BRANCH = "develop"
DRIVE_ROOT = "/content/drive/MyDrive/clear_vision"
MERGED_ZIP = f"{DRIVE_ROOT}/datasets/merged.zip"
MODEL = "yolo11x.pt"  # most accurate COCO model, downloaded to models/ in the repo clone
CONF = 0.5  # add a box only at this confidence or higher
IOU = 0.5  # skip a box that overlaps an existing box of the same class above this IoU
IMGSZ = 640
BATCH = 32
COCO_SOURCES = ["coco_subset", "openimages"]  # these already label the COCO classes: skipped
FORCE_PSEUDO = False  # True: restore the originals from labels_original.zip, then label again

## 1. Setup

In [ ]:
# 1. Setup: Drive, repo, packages
import logging
import subprocess
import sys
from pathlib import Path

import torch
from google.colab import drive

drive.mount("/content/drive")

REPO_DIR = Path("/content/clear-vision")
if (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "-q", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "-q", "-B", BRANCH, f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
commit = subprocess.run(["git", "-C", str(REPO_DIR), "log", "--oneline", "-1"], capture_output=True, text=True).stdout
print(f"Repo at {REPO_DIR}: {commit.strip()}")

%pip install -q ultralytics

sys.path.insert(0, str(REPO_DIR))
import scripts.merge_datasets as merge_datasets  # noqa: E402
import scripts.pseudo_label as pl  # noqa: E402
from src.colab_data import prepare_dataset  # noqa: E402

logging.basicConfig(level=logging.INFO, format="%(message)s", force=True)

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU: yolo11x on CPU takes many hours. Runtime -> Change runtime type -> T4 GPU.")

MERGED_ZIP = Path(MERGED_ZIP)
print(f"merged zip:  {MERGED_ZIP}")
print(f"Drive free:  {merge_datasets.gb(merge_datasets.free_space(MERGED_ZIP.parent))}")

## 2. Load the dataset

Copies the verified `merged.zip` to the Colab disk and unpacks it to
`/content/datasets/merged/`. Images are never read from Drive.

In [ ]:
# 2. Unpack merged.zip to /content (prepare_dataset checks merged.done and the image counts)
DATA_YAML = prepare_dataset(MERGED_ZIP)
DATASET = DATA_YAML.parent
CLASSES = merge_datasets.read_names(DATA_YAML)
print(f"{DATA_YAML}: {len(CLASSES)} classes")

## 3. Pseudo-label

Loads yolo11x once and predicts on every non-COCO image. Then it appends the boxes,
writes the backup, CSV and report, and publishes the zip. It prints the table of added
boxes per source and class.

In [ ]:
# 3. Add pseudo-labels and publish the verified zip
GRID = Path("/content/pseudo_labels_grid.jpg")
ENTRY = pl.pseudo_label(
    MERGED_ZIP, DATASET, model_path=REPO_DIR / "models" / MODEL, conf=CONF, iou=IOU, imgsz=IMGSZ,
    batch=BATCH, coco_sources=COCO_SOURCES, force=FORCE_PSEUDO, grid_path=GRID,
)
if ENTRY is None:
    print("Already pseudo-labeled: nothing changed. Set FORCE_PSEUDO = True to redo it.")

## 4. Boxes added per source and class

In [ ]:
# 4. The table from merge_report.json (also shown when section 3 skipped)
import json

import pandas as pd

report = json.loads(MERGED_ZIP.with_name("merge_report.json").read_text())
entry = report.get("pseudo_labels")
if entry is None:
    print("merge_report.json has no pseudo-labels yet.")
else:
    print(f"created {entry['created']} with {entry['model']}, conf >= {entry['conf']}, IoU > {entry['iou']}")
    print(f"{entry['images_checked']} images checked, {entry['label_files_changed']} label files changed, "
          f"{entry['boxes_added']} boxes added, {entry['skipped_overlap']} skipped as overlapping a label\n")
    print(pl.format_added_table(entry["added"], CLASSES))
    by_split = pd.DataFrame(entry["added_by_split"]).fillna(0).astype(int)
    display(by_split.rename_axis("class").style.set_caption("Boxes added per split"))

## 5. Look at 16 random changed images

Original boxes are **green** and added boxes are **yellow** (with confidence). Check that
the yellow boxes are real objects. If too many are wrong, raise `CONF` and re-run with
`FORCE_PSEUDO = True`.

In [ ]:
# 5. Grid: original boxes green, added boxes yellow
import cv2
import matplotlib.pyplot as plt

if ENTRY is None and MERGED_ZIP.with_name(pl.CSV_FILE).exists():  # skipped run: rebuild from the CSV
    changes = pl.changes_from_csv(DATASET, MERGED_ZIP.with_name(pl.CSV_FILE), CLASSES)
    grid = pl.make_grid(DATASET, changes, CLASSES)
    if grid is not None:
        cv2.imwrite(str(GRID), grid)
if GRID.exists():
    plt.figure(figsize=(20, 12))
    plt.imshow(cv2.cvtColor(cv2.imread(str(GRID)), cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.show()
else:
    print("No changed images to show.")

## 6. Check the published zip

`prepare_dataset()` checks `merged.done`, the image counts and Ultralytics'
`check_det_dataset` on the new zip. It reuses the unpack from section 3.

In [ ]:
# 6. Verify: the new merged.zip still passes prepare_dataset + check_det_dataset
DATA_YAML = prepare_dataset(MERGED_ZIP)
print(MERGED_ZIP.with_suffix(".done").read_text())
print(f"OK: {DATA_YAML}")